In [1]:
import os
import sys
import numpy as np
import pandas as pd
import plotly.express as px
import copy

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, accuracy_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

import warnings
warnings.filterwarnings("ignore")

In [2]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
INPUT_DIR = "../../data/embedded"
LABELS_PATH = "../../data/preprocessed/cleaned_labels.csv"
OUTPUT_DIR = "../../data/results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [5]:
# LOAD DATA
def load_data_recursive(input_dir):
    embeddings_list = []
    filenames = []

    print(f"Scanning '{input_dir}' for embeddings...")

    files_found = 0
    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.endswith(".npy"):
                file_path = os.path.join(root, file)

                try:
                    data = np.load(file_path)

                    if data.ndim == 0:
                        continue
                    elif data.ndim == 2:
                        data = data.squeeze()

                    if data.shape != (1024,):
                        continue

                    embeddings_list.append(data)
                    filenames.append(file.replace(".npy", ""))
                    files_found += 1

                except Exception as e:
                    print(f"Error loading {file}: {e}")

    if files_found == 0:
        return None, None

    print(f"Found {files_found} valid files.")

    X = np.vstack(embeddings_list)

    if not os.path.exists(INPUT_DIR):
        print(f"Error: Directory '{INPUT_DIR}' not found.")
        sys.exit(0)
    
    if X is None:
        print("No valid embeddings found. Check your directory.")
        sys.exit(0)
    
    print(f"Final Matrix Shape: {X.shape} (Samples: {X.shape[0]}, Features: {X.shape[1]})")
    return X, filenames

In [6]:
print("1. Loading data...")
X, filenames = load_data_recursive(INPUT_DIR)

1. Loading data...
Scanning '../../data/embedded' for embeddings...
Found 16272 valid files.
Final Matrix Shape: (16272, 1024) (Samples: 16272, Features: 1024)


In [7]:
print("1.5 Loading labels to balance the dataset...")
df_data = pd.DataFrame(X)
df_data["ecg_id"] = np.array(filenames).astype(int)

true_labels_df = pd.read_csv(LABELS_PATH)

df_merged = df_data.merge(true_labels_df[["ecg_id", "label"]], on="ecg_id", how="inner")
df_merged.dropna(subset="label")

print("\nOriginal class distribution:")
print(df_merged["label"].value_counts())

min_class_size = df_merged["label"].value_counts().min()

df_balanced = df_merged.groupby("label").sample(n=min_class_size, random_state=RANDOM_STATE)

print("\nBalanced class distribution:")
print(df_balanced["label"].value_counts())

X_balanced = df_balanced.drop(columns=["ecg_id", "label"]).values

filenames_balanced = df_balanced["ecg_id"].astype(str).str.zfill(5).tolist()

print(f"\nProceeding with {len(X_balanced)} perfectly balanced samples...")

1.5 Loading labels to balance the dataset...

Original class distribution:
label
NORM    9083
MI      2538
STTC    2406
CD      1709
HYP      536
Name: count, dtype: int64

Balanced class distribution:
label
CD      536
HYP     536
MI      536
NORM    536
STTC    536
Name: count, dtype: int64

Proceeding with 2680 perfectly balanced samples...


In [8]:
X_full = df_merged.drop(columns=["ecg_id", "label"]).values
true_labels_full = df_merged["label"].values

true_labels_balanced = df_balanced["label"].values

print("2. Scaling features...")
scaler_full = StandardScaler()
X_scaled_full = scaler_full.fit_transform(X_full)

2. Scaling features...


In [9]:
print("2. Scaling features...")
scaler_full = StandardScaler()
X_scaled_full = scaler_full.fit_transform(X_full)

scaler_balanced = StandardScaler()
X_scaled_balanced = scaler_balanced.fit_transform(X_balanced)

2. Scaling features...


### Copy
Until this point the notebook matches the ml clustering notebook except normalization.

In [10]:
class DEC_Autoencoder(nn.Module):
    def __init__(self):
        super(DEC_Autoencoder, self).__init__()

        self.encoder = nn.Sequential(
            nn.Linear(1024, 500),
            nn.BatchNorm1d(500),
            nn.ReLU(),
            nn.Linear(500, 100),
            nn.ReLU(),
            nn.BatchNorm1d(100),
            nn.Linear(100, 10)
        )

        self.decoder = nn.Sequential(
            nn.Linear(10, 100),
            nn.BatchNorm1d(100),
            nn.ReLU(),
            nn.Linear(100, 500),
            nn.BatchNorm1d(500),
            nn.ReLU(),
            nn.Linear(500, 1024)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        encoded = F.normalize(encoded, p = 2, dim = 1)
        decoded = self.decoder(encoded)
        return encoded, decoded

In [11]:
autoencoder = DEC_Autoencoder().to(device)
print(autoencoder)

DEC_Autoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=1024, out_features=500, bias=True)
    (1): BatchNorm1d(500, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=500, out_features=100, bias=True)
    (4): ReLU()
    (5): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): Linear(in_features=100, out_features=10, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=10, out_features=100, bias=True)
    (1): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=100, out_features=500, bias=True)
    (4): BatchNorm1d(500, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Linear(in_features=500, out_features=1024, bias=True)
  )
)


In [12]:
X_tensor_balanced = torch.tensor(X_scaled_balanced, dtype=torch.float32).to(device)

dataset = TensorDataset(X_tensor_balanced)
dataloader = DataLoader(dataset, batch_size=256, shuffle=True)

print(f"Data ready! Passing {len(X_tensor_balanced)} patients in batches of 256.")

Data ready! Passing 2680 patients in batches of 256.


In [13]:
print("--- Pre-training the Autoencoder ---")
optimizer = optim.Adam(autoencoder.parameters(), lr=0.001)
criterion = nn.MSELoss()

epochs = 200
autoencoder.train()

for epoch in range(epochs):
    total_loss = 0
    for batch in dataloader:
        batch_data = batch[0]

        encoded, decoded = autoencoder(batch_data)

        loss = criterion(decoded, batch_data)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] | MSE Loss: {total_loss/len(dataloader):.4f}")

print("--- Pre-training Complete! ---")

--- Pre-training the Autoencoder ---


NameError: name 'F' is not defined

In [ ]:
print("--- Extracting 10D Embeddings & Running Kmeans ---")

autoencoder.eval()

with torch.no_grad():
    z_initial, _ = autoencoder(X_tensor_balanced)
    z_initial_np = z_initial.cpu().numpy()

kmeans = KMeans(n_clusters = 80, n_init = 20, random_state = RANDOM_STATE)
initial_kmeans_labels = kmeans.fit_predict(z_initial_np)

initial_centers = torch.tensor(kmeans.cluster_centers_, dtype = torch.float32).to(device)

mapped_preds = np.zeros_like(initial_kmeans_labels, dtype = object)
true_labels_series = pd.Series(true_labels_balanced)

for cluster_id in range(80):
    idx = np.where(initial_kmeans_labels == cluster_id)[0]
    majority_disease = true_labels_series.iloc[idx].mode()[0]
    mapped_preds[idx] = majority_disease

initial_purity = accuracy_score(true_labels_balanced, mapped_preds)

print(f"Initial cluster centers established!")
print(f"Baseline Purity (Before DEC fine-tuning): {initial_purity*100:.1f}%")

In [ ]:
import torch.nn.functional as F

class DECLayer(nn.Module):
    def __init__(self, n_clusters = 5, hidden_dim = 10, alpha = 1.0):
        super(DECLayer, self).__init__()
        self.alpha = alpha
        self.cluster_centers = nn.Parameter(torch.Tensor(n_clusters, hidden_dim))

    def forward(self, x):
        dist = torch.sum((x.unsqueeze(1) - self.cluster_centers) ** 2, dim = 2)

        q = 1.0 / (1.0 + dist / self.alpha)
        q = q ** ((self.alpha + 1.0) / 2.0)
        q = (q.t() / torch.sum(q, dim=1)).t()
        return q

class DECModel(nn.Module):
    def __init__(self, pretrained_autoencoder, initial_centers):
        super(DECModel, self).__init__()
        self.encoder = pretrained_autoencoder.encoder
        self.dec_layer = DECLayer(n_clusters=5, hidden_dim=10)
        self.dec_layer.cluster_centers.data = initial_centers

    def forward(self, x):
        z = self.encoder(x)
        q = self.dec_layer(z)
        return z, q

In [ ]:
def target_distribution(q):
    weight = q ** 2 / torch.sum(q, dim = 0)
    return (weight.t() / torch.sum(weight, dim = 1)).t()

clean_autoencoder_dec = copy.deepcopy(autoencoder)
dec_model = DECModel(clean_autoencoder_dec, initial_centers.clone()).to(device)

optimizer_dec = optim.SGD(dec_model.parameters(), lr = 0.01, momentum = 0.9) # optim.Adam(dec_model.parameters(), lr = 1e-4)
criterion_dec = nn.KLDivLoss(reduction = "batchmean")

dataloader = DataLoader(dataset, batch_size=256, shuffle=False)

In [ ]:
print("--- Starting Deep Embedded Clustering (DEC) ---")
epochs_dec = 150
update_interval = 5

dec_model.train()

with torch.no_grad():
    _, q_all = dec_model(X_tensor_balanced)
    p_all = target_distribution(q_all)

global_batch_count = 0

for epoch in range(epochs_dec):
    total_loss = 0

    for batch_idx, batch in enumerate(dataloader):
        batch_data = batch[0]

        if global_batch_count > 0 and global_batch_count % update_interval == 0:
            with torch.no_grad():
                _, q_all_updated = dec_model(X_tensor_balanced)
                p_all = target_distribution(q_all_updated)

        batch_start = batch_idx * dataloader.batch_size
        batch_end = min((batch_idx + 1) * dataloader.batch_size, len(X_tensor_balanced))
        p_batch = p_all[batch_start:batch_end]

        _, q_batch = dec_model(batch_data)

        loss = criterion_dec(torch.log(q_batch + 1e-8), p_batch)

        optimizer_dec.zero_grad()
        loss.backward()
        optimizer_dec.step()
        
        total_loss += loss.item()
        global_batch_count += 1

    if (epoch + 1) % 10 == 0:
        print(f"DEC Epoch [{epoch+1}/{epochs_dec}] | KL Loss: {total_loss/len(dataloader):.4f}")

print("--- Deep Clustering Complete! ---")

In [ ]:
print("--- Final DEC Model Metrics ---")
dec_model.eval()

with torch.no_grad():
    z_final, q_final = dec_model(X_tensor_balanced)

    final_labels = torch.argmax(q_final, dim = 1).cpu().numpy()

mapped_preds_dec = np.zeros_like(final_labels, dtype=object)
true_labels_series = pd.Series(true_labels_balanced)

for cluster_id in range(80):
    idx = np.where(final_labels == cluster_id)[0]
    
    if len(idx) > 0:
        majority_disease = true_labels_series.iloc[idx].mode()[0]
        mapped_preds_dec[idx] = majority_disease

final_dec_purity = accuracy_score(true_labels_balanced, mapped_preds_dec)
ari_dec = adjusted_rand_score(true_labels_balanced, final_labels)

print(f"Clusters Found: 5")
print(f"Adjusted Rand Index (ARI): {ari_dec:.4f}")
print(f"Cluster Purity Accuracy:   {final_dec_purity*100:.1f}%")
print("========================================")

In [ ]:
class DECLayer(nn.Module):
    def __init__(self, n_clusters = 5, hidden_dim = 10, alpha = 1.0):
        super(DECLayer, self).__init__()
        self.alpha = alpha
        self.cluster_centers = nn.Parameter(torch.Tensor(n_clusters, hidden_dim))

    def forward(self, x):
        dist = torch.sum((x.unsqueeze(1) - self.cluster_centers) ** 2, dim = 2)

        q = 1.0 / (1.0 + dist / self.alpha)
        q = q ** ((self.alpha + 1.0) / 2.0)
        q = (q.t() / torch.sum(q, dim=1)).t()
        return q

class IDECModel(nn.Module):
    def __init__(self, pretrained_autoencoder, initial_centers):
        super(IDECModel, self).__init__()
        self.encoder = pretrained_autoencoder.encoder
        self.decoder = pretrained_autoencoder.decoder
        self.dec_layer = DECLayer(n_clusters=80, hidden_dim=10)
        self.dec_layer.cluster_centers.data = initial_centers

    def forward(self, x):
        z = self.encoder(x)
        decoded = self.decoder(z)
        q = self.dec_layer(z)
        return z, decoded, q

In [ ]:
def target_distribution(q):
    weight = q ** 2 / torch.sum(q, dim = 0)
    return (weight.t() / torch.sum(weight, dim = 1)).t()

clean_autoencoder_idec = copy.deepcopy(autoencoder)
idec_model = IDECModel(clean_autoencoder_idec, initial_centers.clone()).to(device)

optimizer_idec = optim.Adam(idec_model.parameters(), lr = 5e-4) # optim.SGD(idec_model.parameters(), lr = 0.01, momentum = 0.9)
criterion_kl = nn.KLDivLoss(reduction = "batchmean")
criterion_mse = nn.MSELoss()

dataloader = DataLoader(dataset, batch_size=256, shuffle=False)

In [ ]:
print("--- Starting IDEC ---")
epochs_idec = 100
gamma = 0.8
update_interval = 11

idec_model.train()

with torch.no_grad():
    _, _, q_all = idec_model(X_tensor_balanced)
    p_all = target_distribution(q_all)

global_batch_count = 0

for epoch in range(epochs_idec):
    total_kl_loss = 0
    total_mse_loss = 0

    for batch_idx, batch in enumerate(dataloader):
        batch_data = batch[0]

        if global_batch_count > 0 and global_batch_count % update_interval == 0:
            with torch.no_grad():
                _, _, q_all_updated = idec_model(X_tensor_balanced)
                p_all = target_distribution(q_all_updated)

        batch_start = batch_idx * dataloader.batch_size
        batch_end = min((batch_idx + 1) * dataloader.batch_size, len(X_tensor_balanced))
        p_batch = p_all[batch_start:batch_end]

        z, decoded, q_batch = idec_model(batch_data)

        loss_kl = criterion_kl(torch.log(q_batch + 1e-8), p_batch)
        loss_mse = criterion_mse(decoded, batch_data)

        loss = loss_kl + (gamma * loss_mse)

        optimizer_idec.zero_grad()
        loss.backward()
        optimizer_idec.step()
        
        total_kl_loss += loss_kl.item()
        total_mse_loss += loss_mse.item()
        global_batch_count += 1

    if (epoch + 1) % 10 == 0:
        print(f"IDEC Epoch [{epoch+1}/{epochs_idec}] | KL: {total_kl_loss/len(dataloader):.4f} | MSE: {total_mse_loss/len(dataloader):.4f}")

print("--- IDEC Complete! ---")

In [ ]:
print("--- Final IDEC Model Metrics ---")
idec_model.eval()

with torch.no_grad():
    z_final, _, q_final = idec_model(X_tensor_balanced)

    final_labels = torch.argmax(q_final, dim = 1).cpu().numpy()

mapped_preds_dec = np.zeros_like(final_labels, dtype=object)
true_labels_series = pd.Series(true_labels_balanced)

for cluster_id in range(80):
    idx = np.where(final_labels == cluster_id)[0]
    
    if len(idx) > 0:
        majority_disease = true_labels_series.iloc[idx].mode()[0]
        mapped_preds_dec[idx] = majority_disease

final_idec_purity = accuracy_score(true_labels_balanced, mapped_preds_dec)
ari_dec = adjusted_rand_score(true_labels_balanced, final_labels)

print(f"Clusters Found: n")
print(f"Adjusted Rand Index (ARI): {ari_dec:.4f}")
print(f"Cluster Purity Accuracy:   {final_idec_purity*100:.1f}%")
print("========================================")